##Imports

In [24]:
#Data Manipulation and Visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Data Preprocessing

from sklearn.preprocessing import OneHotEncoder,LabelEncoder, OrdinalEncoder,TargetEncoder
from sklearn.preprocessing import StandardScaler,MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

#Pipeline

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

#Model Building

from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor

#Evaluation Metrics

from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score

In [25]:
#Importing the cleaned data

df=pd.read_csv('../data/youtube_df_cleaned (3).csv')

In [26]:
#Initial health check of the dataset

df.shape

#(120000,13)

(120000, 13)

In [27]:
df.head()

,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,category,device,country,ad_revenue_usd,comment_rate,engagement_rate,watch_time_per_view
0,9936,1221,320,26497.214184,2.862137,228086,Entertainment,TV,IN,203.178237,0.032206,0.155093,2.666789
1,10017,642,346,15209.747445,23.738069,736015,Gaming,Tablet,CA,140.880508,0.034541,0.098632,1.518393
2,10097,1979,187,57332.658498,26.200634,240534,Education,TV,CA,360.134008,0.018520,0.214519,5.678187
3,10034,1191,242,31334.517771,11.770340,434482,Entertainment,Mobile,UK,224.638261,0.024118,0.142814,3.122834
4,9889,1858,477,15665.666434,6.635854,42030,Education,Mobile,CA,165.514388,0.048235,0.236121,1.584151


In [28]:
df.isna().sum()

views                   0
likes                   0
comments                0
watch_time_minutes      0
video_length_minutes    0
subscribers             0
category                0
device                  0
country                 0
ad_revenue_usd          0
comment_rate            0
engagement_rate         0
watch_time_per_view     0
dtype: int64

In [29]:
df.duplicated().sum()

0

In [30]:
#Derived features introduced multicollinearity due to mathematical dependency,
#so I retained base features to maintain model stability

df_model = df.drop(['comment_rate','watch_time_per_view'],axis=1)
df_model.shape

(120000, 11)

In [31]:
df_model.head()

,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,category,device,country,ad_revenue_usd,engagement_rate
0,9936,1221,320,26497.214184,2.862137,228086,Entertainment,TV,IN,203.178237,0.155093
1,10017,642,346,15209.747445,23.738069,736015,Gaming,Tablet,CA,140.880508,0.098632
2,10097,1979,187,57332.658498,26.200634,240534,Education,TV,CA,360.134008,0.214519
3,10034,1191,242,31334.517771,11.770340,434482,Entertainment,Mobile,UK,224.638261,0.142814
4,9889,1858,477,15665.666434,6.635854,42030,Education,Mobile,CA,165.514388,0.236121


##Splitting Feature Columns(X) and Target Column(Y)

In [32]:
df_model.columns

Index(['views', 'likes', 'comments', 'watch_time_minutes',
       'video_length_minutes', 'subscribers', 'category', 'device', 'country',
       'ad_revenue_usd', 'engagement_rate'],
      dtype='object')

In [33]:
#Splitting X and Y

X=df_model.drop(['ad_revenue_usd'],axis=1)
Y=df_model['ad_revenue_usd']

In [34]:
X

,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,category,device,country,engagement_rate
0,9936,1221,320,26497.214184,2.862137,228086,Entertainment,TV,IN,0.155093
1,10017,642,346,15209.747445,23.738069,736015,Gaming,Tablet,CA,0.098632
2,10097,1979,187,57332.658498,26.200634,240534,Education,TV,CA,0.214519
3,10034,1191,242,31334.517771,11.770340,434482,Entertainment,Mobile,UK,0.142814
4,9889,1858,477,15665.666434,6.635854,42030,Education,Mobile,CA,0.236121
...,...,...,...,...,...,...,...,...,...,...
119995,9853,1673,147,42075.704885,25.490195,210818,Education,Tablet,US,0.184715
119996,10128,1709,63,57563.703040,16.229133,878860,Music,Desktop,UK,0.174961
119997,10267,700,0,27549.714659,23.822365,576756,Tech,Tablet,CA,0.068182
119998,10240,1616,106,56967.384382,7.753099,585138,Music,Mobile,UK,0.168164


In [35]:
Y

0         203.178237
1         140.880508
2         360.134008
3         224.638261
4         165.514388
             ...    
119995    280.986396
119996    354.612981
119997    203.643106
119998    351.525811
119999    253.842824
Name: ad_revenue_usd, Length: 120000, dtype: float64

##Splitting Train and Test Data



In [36]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)
X_train.shape,X_test.shape,Y_train.shape,Y_test.shape

((96000, 10), (24000, 10), (96000,), (24000,))

In [37]:
df_model.columns

Index(['views', 'likes', 'comments', 'watch_time_minutes',
       'video_length_minutes', 'subscribers', 'category', 'device', 'country',
       'ad_revenue_usd', 'engagement_rate'],
      dtype='object')

In [38]:
Num_Col = X.select_dtypes(include=['int64','float64'])
Num_Col

,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,engagement_rate
0,9936,1221,320,26497.214184,2.862137,228086,0.155093
1,10017,642,346,15209.747445,23.738069,736015,0.098632
2,10097,1979,187,57332.658498,26.200634,240534,0.214519
3,10034,1191,242,31334.517771,11.770340,434482,0.142814
4,9889,1858,477,15665.666434,6.635854,42030,0.236121
...,...,...,...,...,...,...,...
119995,9853,1673,147,42075.704885,25.490195,210818,0.184715
119996,10128,1709,63,57563.703040,16.229133,878860,0.174961
119997,10267,700,0,27549.714659,23.822365,576756,0.068182
119998,10240,1616,106,56967.384382,7.753099,585138,0.168164


In [39]:
Cat_Col = X.select_dtypes(include='object')
Cat_Col

,category,device,country
0,Entertainment,TV,IN
1,Gaming,Tablet,CA
2,Education,TV,CA
3,Entertainment,Mobile,UK
4,Education,Mobile,CA
...,...,...,...
119995,Education,Tablet,US
119996,Music,Desktop,UK
119997,Tech,Tablet,CA
119998,Music,Mobile,UK


In [40]:
#Numerical Transformer

Num_trans = Pipeline(steps=[
    ('Imputing',SimpleImputer(strategy='median')),
    ('Scaling',MinMaxScaler()),
])

#Categorical tranformer

Cat_trans = Pipeline(steps=[
    ('Imputing',SimpleImputer(strategy='most_frequent')),
    ('Encoding',OneHotEncoder())
])

In [41]:
#Column Transformer

Pre_Process_Step = ColumnTransformer(transformers=[
    ('Numerical_Transformer',Num_trans,Num_Col.columns),
    ('Categorical_Transformer',Cat_trans,Cat_Col.columns),
])

Pre_Process_Step

ColumnTransformer(transformers=[('Numerical_Transformer',
                                 Pipeline(steps=[('Imputing',
                                                  SimpleImputer(strategy='median')),
                                                 ('Scaling', MinMaxScaler())]),
                                 Index(['views', 'likes', 'comments', 'watch_time_minutes',
       'video_length_minutes', 'subscribers', 'engagement_rate'],
      dtype='object')),
                                ('Categorical_Transformer',
                                 Pipeline(steps=[('Imputing',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('Encoding',
                                                  OneHotEncoder())]),
                                 Index(['category', 'device', 'country'], dtype='object'))])

## 1) Linear Regression Model

In [42]:
#Pipeline

LR_Pipeline = Pipeline(steps=[
    ('Pre_Processing',Pre_Process_Step),
    ('Linear_Regression',LinearRegression())
])

LR_Pipeline

Pipeline(steps=[('Pre_Processing',
                 ColumnTransformer(transformers=[('Numerical_Transformer',
                                                  Pipeline(steps=[('Imputing',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('Scaling',
                                                                   MinMaxScaler())]),
                                                  Index(['views', 'likes', 'comments', 'watch_time_minutes',
       'video_length_minutes', 'subscribers', 'engagement_rate'],
      dtype='object')),
                                                 ('Categorical_Transformer',
                                                  Pipeline(steps=[('Imputing',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('Encoding',
                                                                   OneHotEncoder())]),
                                                  Index(['category', 'device', 'country'], dtype='object'))])),
                ('Linear_Regression', LinearRegression())])

In [43]:
LR_Pipeline.fit(X_train,Y_train)

Pipeline(steps=[('Pre_Processing',
                 ColumnTransformer(transformers=[('Numerical_Transformer',
                                                  Pipeline(steps=[('Imputing',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('Scaling',
                                                                   MinMaxScaler())]),
                                                  Index(['views', 'likes', 'comments', 'watch_time_minutes',
       'video_length_minutes', 'subscribers', 'engagement_rate'],
      dtype='object')),
                                                 ('Categorical_Transformer',
                                                  Pipeline(steps=[('Imputing',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('Encoding',
                                                                   OneHotEncoder())]),
                                                  Index(['category', 'device', 'country'], dtype='object'))])),
                ('Linear_Regression', LinearRegression())])

In [44]:
#Prediction

Y_train_pred = LR_Pipeline.predict(X_train)
Y_test_pred = LR_Pipeline.predict(X_test)



In [45]:
#Model Evaluation

#MAE for Train and Test data

MAE_train = mean_absolute_error(Y_train,Y_train_pred)
MAE_test=mean_absolute_error(Y_test,Y_test_pred)
print('Train MAE',MAE_train)
print('Test MAE',MAE_test)
print('\n')

#MSE fro train and test data

MSE_train = mean_squared_error(Y_train,Y_train_pred)
MSE_test = mean_squared_error(Y_test,Y_test_pred)
#print('Train MSE',MSE_train)
#print('Test MSE',MSE_test)
#print('\n')

#RMSE for train and test data

RMSE_train = np.sqrt(MSE_train)
RMSE_test = np.sqrt(MSE_test)
print('Train RMSE',RMSE_train)
print('Test RMSE',RMSE_test)



Train MAE 3.514096410289347
Test MAE 3.3662552147538505


Train RMSE 13.926980336968775
Test RMSE 13.537711597029427


In [46]:
#Performance Metric

#R2 Score

R2_train = r2_score(Y_train,Y_train_pred)
R2_test = r2_score(Y_test,Y_test_pred)
print('Train R2 Score',R2_train)
print('Test R2 Score',R2_test)


Train R2 Score 0.9494891553992556
Test R2 Score 0.952167344449049


My LinearRegression model achieved around 95% R² on both train and test data, indicating strong performance with no overfitting, and low MAE shows accurate predictions.

In [47]:
LR_Pipeline.steps[1][1].coef_

array([ 4.17772274e+00,  4.71567344e+01,  1.18128614e+01,  2.21150162e+02,
        7.17708920e-02,  7.32005170e-02, -1.92641732e+01, -1.66263649e+13,
       -1.66263649e+13, -1.66263649e+13, -1.66263649e+13, -1.66263649e+13,
       -1.66263649e+13,  2.39737884e+12,  2.39737884e+12,  2.39737884e+12,
        2.39737884e+12, -7.20116054e+12, -7.20116054e+12, -7.20116054e+12,
       -7.20116054e+12, -7.20116054e+12, -7.20116054e+12])

In [48]:
# Feature names after preprocessing
feature_names = LR_Pipeline.named_steps['Pre_Processing'].get_feature_names_out()

# Coefficients
coefficients = LR_Pipeline.named_steps['Linear_Regression'].coef_

# Dataframe
import pandas as pd

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Absolute values
coef_df['abs_coef'] = coef_df['Coefficient'].abs()

# Sorting by importance
coef_df = coef_df.sort_values(by='abs_coef', ascending=False)

# Top features
coef_df.head(10)

,Feature,Coefficient,abs_coef
9,Categorical_Transformer__category_Gaming,-1.662636e+13,1.662636e+13
11,Categorical_Transformer__category_Music,-1.662636e+13,1.662636e+13
10,Categorical_Transformer__category_Lifestyle,-1.662636e+13,1.662636e+13
8,Categorical_Transformer__category_Entertainment,-1.662636e+13,1.662636e+13
7,Categorical_Transformer__category_Education,-1.662636e+13,1.662636e+13
12,Categorical_Transformer__category_Tech,-1.662636e+13,1.662636e+13
19,Categorical_Transformer__country_DE,-7.201161e+12,7.201161e+12
22,Categorical_Transformer__country_US,-7.201161e+12,7.201161e+12
21,Categorical_Transformer__country_UK,-7.201161e+12,7.201161e+12
18,Categorical_Transformer__country_CA,-7.201161e+12,7.201161e+12


#Insight :
Watch time is the most important feature, showing a strong positive impact on revenue.
Likes and comments also positively influence revenue, indicating user engagement matters.
Views have a smaller effect, suggesting engagement quality is more important than quantity.
Engagement rate shows a negative impact due to multicollinearity with likes and comments.
Categorical features like device, country, and category have minimal influence compared to behavioral features.

## 2) Ridge Regression Model

In [49]:
from sklearn.linear_model import Ridge

ridge_pipeline = Pipeline(steps=[
    ('Preprocessing', Pre_Process_Step),
    ('Model', Ridge(alpha=0.9))
])

# Train
ridge_pipeline.fit(X_train, Y_train)

# Predict
Y_train_pred = ridge_pipeline.predict(X_train)
Y_test_pred = ridge_pipeline.predict(X_test)

# Evaluate

print("Train R2:", r2_score(Y_train, Y_train_pred))
print("Test R2:", r2_score(Y_test, Y_test_pred))

Train R2: 0.9494883564615143
Test R2: 0.9521721414906903





*   The Ridge Regression model achieved an R² score of ~0.95 on both train and test data, indicating strong predictive performance. The similar scores suggest that the model is well-generalized and not overfitting.
*   Regularization helped stabilize the coefficients without sacrificing accuracy.



## 3) Lasso Regression



In [50]:
from sklearn.linear_model import Lasso

lasso_pipeline = Pipeline(steps=[
    ('Preprocessing', Pre_Process_Step),
    ('Model', Lasso(alpha=0.1))
])

# Train
lasso_pipeline.fit(X_train, Y_train)

# Predict
Y_train_pred = lasso_pipeline.predict(X_train)
Y_test_pred = lasso_pipeline.predict(X_test)

# Evaluate

print("Train R2:", r2_score(Y_train, Y_train_pred))
print("Test R2:", r2_score(Y_test, Y_test_pred))

Train R2: 0.9492930608284973
Test R2: 0.9520359727029105




*   Lasso Regression achieved an R² score of ~0.95 on both train and test data, indicating strong predictive performance.
*   It also helps in feature selection by shrinking less important feature coefficients, making the model more interpretable.







*   Even though Linear Regression performed well, I applied Ridge and Lasso to handle multicollinearity and compare model robustness.
*    This helps ensure stability and better feature selection



## 4) Random Forest Regressor

In [51]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('Preprocessing', Pre_Process_Step),
    ('Model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train
rf_pipeline.fit(X_train, Y_train)

# Predict
Y_train_pred = rf_pipeline.predict(X_train)
Y_test_pred = rf_pipeline.predict(X_test)

# Evaluate

print("Train R2:", r2_score(Y_train, Y_train_pred))
print("Test R2:", r2_score(Y_test, Y_test_pred))

Train R2: 0.9925657981053231
Test R2: 0.9490237440626076




*   Random Forest achieved a very high R² score (~0.99 on train and ~0.95 on test), indicating strong predictive performance
*   The slight drop from train to test suggests minor overfitting, but overall the model generalizes well.



## 5) Gradient Boosting Regressor

In [43]:
from sklearn.ensemble import GradientBoostingRegressor

gb_pipeline = Pipeline(steps=[
    ('Preprocessing', Pre_Process_Step),
    ('Model', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
])

# Train
gb_pipeline.fit(X_train, Y_train)

# Predict
Y_train_pred = gb_pipeline.predict(X_train)
Y_test_pred = gb_pipeline.predict(X_test)

# Evaluate

print("Train R2:", r2_score(Y_train, Y_train_pred))
print("Test R2:", r2_score(Y_test, Y_test_pred))

Train R2: 0.9500539506594086
Test R2: 0.9520247254570229



*   Gradient Boosting achieved consistent R² scores (~0.95) on both train and test data, indicating strong and stable performance.
*   The nearly equal scores show that the model generalizes well with minimal overfitting.



Although Random Forest achieved a higher train R², it showed slight overfitting compared to Linear/Ridge models. Linear Regression and Ridge provided almost equal and high performance (~95% R²) on both train and test data, indicating better generalization and stability. Therefore, a simpler linear model is preferred for this problem.

Since the goal is to predict ad revenue and understand feature impact, Linear Regression is preferred due to its interpretability. It clearly shows how each feature contributes to revenue, which is useful for business decision-making. Additionally, it provides stable and accurate predictions without overfitting.

In [52]:
import joblib

joblib.dump(LR_Pipeline, '../app/model_pipeline.pkl')

['../app/model_pipeline.pkl']